In [1]:
#Import packages for calculation
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pynucastro as pyna

In [ ]:
class MainphaseStar:
    #Stellar evolution simulator that opperates on the core conceit that the star remains in the Main Phase (where the primary energy in the star comes from hydrogen fusion) and remain in thermal and hydrostatic equilibrium
    
    #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
    ppchainrate = pyna.load_rate("p(p,)d")
    cnochainrate = pyna.load_rate("n14(p,g)o15")

    #radiation constant as defined by Stefan Bolzman Law
    a = 7.5657 * 10 **(-16)
    #units j (m**-3) (K**-4)

    def __init__(self, temp, dens, pres, pcenthydrogen, radius):
        #Takes 4 1d-arrays of equal size and 1 float.64
        self.t = temp #in kelvin
        self.d = dens #in gram/meters**3
        self.p = pres 
        self.pcenthydrogen = pcenthydrogen #percent by mass of hydrogen for each layer
        self.r = radius #in meters
        #obtains the number of cells as well as the size of each cell 
        self.n = len(temp)
        self.rinc = self.r / self.n

        #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
        rl = pyna.full_library()
        self.ppchainrate = rl.get_rate_by_name("p(p,)d")
        self.cnochainrate = rl.get_rate_by_name("n14(p,g)o15")

    def rincrinitialize(self):
        #reinitializes the calculation for the radius increment of each cell
        self.rinc = self.r / self.n
        self.rbycell = (self.n - np.asarray(range(0, self.n)))*self.rinc

    def massbyrad(self):
        #initializes mass array for each cell, where mass is the volume * the density
        self.mass = np.zeros_like(self.temp) #Mass of interior of star from each cell
        self.layermass = np.zeros_like(self.temp) #mass of material in the layer composed by each cell
        for i in range(1, self.n):
            self.mass[-i] = 4 / 3 * np.pi (self.rinc * i) ** 3 * (np.sum(self.dens[-i:])/i)  
        for i in range(1, self.n):
            self.layermass[-i] = self.mass[-i] - np.sum(self.mass[-i-1:])

    def fusionenergy(self, density, perhydro, temp):
        #calculate fusion energy per unit mass based on temp and density of hydrogen- using p-p and c-n-o fusion paths 
        #Note: pynucastro assumes rho density is in grams/cm**3 thus we have to convert   
        
        #evaluates reaction rates per cm^3 
        ppeval = self.ppchainrate.eval(temp, density*perhydro/(10**6))
        cnoeval = self.cnochainrate.eval(temp, density*perhydro/(10**6))

        #convert to reaction rates per unit mass
        ppeval = ppeval*10**6/density
        cnoeval = cnoeval*10**6/density

        #evaluate energy result per unit mass then converts mega electron volt to electron volt
        energyperunitmass = (ppeval*26.72 + cnoeval*25)*1000000 
        return energyperunitmass

        

    def massolve(self, masses):
        #Takes list of t masses to solve system at
        #returns a t by n array for each system quantity of values at each radius interval n for that mass
        #evaluates from center out for each time
        return 

RateFileError: File WindowsPath('p(p,)d') not found in the working directory, c:\Users\Ramie\miniconda3\envs\my-env\Lib\site-packages\pynucastro\library, or c:\Users\Ramie\miniconda3\envs\my-env\Lib\site-packages\pynucastro\library\tabular